> **Meridian AgentOps workshop · notebook 02 of 6 (Modules 4–5).** The reusable code lives in the
> repo's `app/` package (agent, tools, MCP service desk, evaluators, config) — these notebooks
> import it, so a fresh session only needs the bootstrap cells below instead of re-running
> earlier modules. **Prerequisite:** notebook 00 has run once against your Langfuse project.


## Tracing & monitoring agent executions

- you can read any agent trace — spans, tool calls, tokens, cost, latency
- follow a multi-turn session end-to-end, recognize timeout-and-retry patterns
- slice traffic with sessions, users, tags and filters.

In [ ]:
# ── 0.1 Get the code + the pinned stack (fresh Colab VM: clone first) ──
import os
if not os.path.isdir("../app"):                    # fresh Colab VM → clone the repo
    !git clone https://github.com/kartik-nighania/data-hack-summit-2026.git _workshop_repo
    %cd _workshop_repo/workshop
%pip install -q -r ../requirements.txt
print("✅ stack ready — if pip just upgraded packages, do Run ▸ Restart session once and rerun from the top.")


In [ ]:
import os, sys

sys.path.insert(0, os.path.abspath(".."))        # make the repo's app/ package importable

# Jupyter kernels already run an event loop; this lets libraries that call
# asyncio.run()/run_until_complete work inside notebook cells.
import nest_asyncio
nest_asyncio.apply()

print("✅ environment prepared |", sys.version.split()[0])


In [ ]:
# ── 0.2 Your API keys — paste them in, then run. Colab saves your notebook copy to Drive,
#        so this is a one-time edit per notebook (testing keys only — don't share the copy).
import os
os.environ["OPENAI_API_KEY"] = "sk-proj-..."          # platform.openai.com/api-keys
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."       # cloud.langfuse.com → Settings ▸ API Keys
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
os.environ["LANGFUSE_BASE_URL"] = "https://us.cloud.langfuse.com"   # EU account: https://cloud.langfuse.com

from app.config import load_keys
load_keys()   # validates; unedited placeholders fall back to .env or a prompt


In [ ]:
# ── 0.3 Constants (config.yaml) + the Langfuse client (PII masking hook registered) ─
from app.config import trace_url, get_lf
lf = get_lf()
if not lf.auth_check():
    raise SystemExit("❌ Langfuse authentication FAILED. Check keys + LANGFUSE_HOST region "
                     "(EU: https://cloud.langfuse.com / US: https://us.cloud.langfuse.com).")
LANGFUSE_HOST = os.environ["LANGFUSE_HOST"]
import importlib.metadata as _md
print("✅ Langfuse authenticated:", LANGFUSE_HOST)
for p in ["langfuse", "langchain", "langgraph", "deepeval", "openai", "fastmcp"]:
    print(f"   {p}=={_md.version(p)}")


In [ ]:
# Session imports + a warm-up trace for 4.1 to dissect (the monolith reused the Module-3 run)
from app.agent import run_agent
from app.tools import FLAKY_MODE
r = await run_agent("What's my outstanding balance?", "CUST-1002",
                    session_id="sess-m4-warmup", tags=["module-4"])
lf.flush()
print("warm-up trace ready:", r["url"])


In [ ]:
# ── 4.1 Anatomy of a trace, programmatically (same data the UI shows) ────────
import pandas as pd

t = lf.api.trace.get(r["trace_id"])          # r = the module-3 run
rows = []
for o in t.observations:
    ms = (o.end_time - o.start_time).total_seconds() * 1000 if (o.end_time and o.start_time) else None
    rows.append({"type": o.type, "name": o.name, "ms": round(ms) if ms else None,
                 "model": getattr(o, "model", None),
                 "tokens": (o.usage.total if getattr(o, "usage", None) else None),
                 "cost_usd": getattr(o, "calculated_total_cost", None)})
df = pd.DataFrame(rows).sort_values("ms", ascending=False)
display(df.head(12))
print(f"TRACE TOTALS  latency={t.latency:.2f}s  cost=${t.total_cost:.5f}  observations={len(t.observations)}")
print("→ The same numbers, visually: open the trace and read the waterfall + cost per step.", trace_url(t.id))

**Reading the table:** the expensive rows are always the generations (`model` column filled); tool spans are
cheap; the supervisor runs *twice* (route in, then FINISH). Step-level cost attribution is what lets you answer
*"which node is eating my budget?"* — for multi-agent systems that's the difference between guessing and knowing.

In [ ]:
# ── 4.2 One conversation, end-to-end: sessions = the thread id ───────────────
session_id = "sess-m4-priya-emi"          # in a real app: your conversation/thread id (e.g. LangGraph thread_id)
history, last = [], None
for turn in [
    "Hi, I have two loans with you. What's the outstanding on each?",
    "For the top-up loan, what would the next 3 EMIs look like?",
    "Thanks. And if I prepay some of it, are there charges?",
]:
    last = await run_agent(turn, "CUST-1003", user_id="CUST-1003", session_id=session_id,
                     tags=["module-4"], history=history)
    from langchain_core.messages import HumanMessage, AIMessage
    history += [HumanMessage(content=turn), AIMessage(content=last["answer"])]
    print(f"→ turn traced: {last['route']} | {last['latency_s']}s")
lf.flush()
print("\nFinal answer:\n", last["answer"][:400])
print("\n✅ Open Langfuse ▸ Sessions ▸", session_id, "— all three traces stitched into one conversation view.")

### ✅ CHECKPOINT — sessions & users
- **Sessions view** (`Tracing ▸ Sessions`): `sess-m4-priya-emi` shows the 3-turn conversation with combined
  cost/latency — this is your conversation-level debugging view, and later the unit for *conversation-level scores*.
- **Users view** (`Tracing ▸ Users`): `CUST-1003` now has traces, total cost, and activity over time. Because
  `run_agent` maps `user_id` on every trace, you get **per-user dashboards for free** — who are the heaviest
  users, whose sessions fail, what does one customer cost you per month.

In [ ]:
# ── 4.3 Timeouts & retries — what resilience looks like in a trace ───────────
FLAKY_MODE["on"] = True         # the core-banking EMI service now times out ~60% of calls
flaky = await run_agent("When is my next EMI due and how much of it is interest?", "CUST-1004",
                  session_id="sess-m4-flaky", tags=["module-4", "flaky-demo"])
FLAKY_MODE["on"] = False
lf.flush()
print(flaky["answer"][:300])
print("\n🔗", flaky["url"])
print("Open the trace → inside the get_emi_schedule TOOL span you'll see emi-fetch-attempt-1/2/3 spans;")
print("failed attempts are marked level=ERROR with the timeout message, then a later attempt succeeds.")

In [ ]:
# ── 4.4 Organizing traffic: tags, environments, and the filter bar ───────────
for q, cid, tags in [
    ("What are the charges for foreclosing my fixed-rate loan?", "CUST-1002", ["module-4", "billing"]),
    ("I want to file a complaint about a wrongly charged late fee.", "CUST-1005", ["module-4", "priority"]),
]:
    out = await run_agent(q, cid, session_id=f"sess-m4-{cid}", tags=tags)
    print(f"traced {out['route']} tags={tags}")
lf.flush()
print("""
Try these in the Tracing table (top filter bar):
  · Tags        include  'priority'          → the complaint run
  · Level       is       ERROR               → the flaky-demo trace (failed attempts bubble up)
  · User ID     is       CUST-1003           → Priya's session turns
  · Environment is       development         → everything so far (production arrives in Module 7)
  · Session     click any trace → 'Session' link → the full conversation
""")

**Environments** are the clean way to split `development` from `production` telemetry in one project — we
initialized the client with `environment="development"`, and the Module 7 traffic generator will stamp
`production` per-request via `propagate_attributes(environment=...)`. Same code, clean separation in every
view, metric and evaluator filter.

## Module 5 · Capturing user feedback

🎯 *Outcome: real user signals (👍/👎 + a 1–5 rating) flow from "your app" into Langfuse as scores, attached to
exactly the right trace — even when the feedback arrives minutes later.*

The engineering problem is **correlation**: when the user clicks 👎, your feedback endpoint only knows your
app's request id — not Langfuse's trace id. The clean solution: **derive the trace id deterministically from
the request id** (`create_trace_id(seed=…)`), so any part of your system can score the trace later without
storing a mapping. Feedback becomes a **score**: Langfuse recommends thumbs as a BOOLEAN score named
`user-feedback` (1/0), with an idempotency `score_id` so re-votes overwrite instead of duplicating.

In [ ]:
# ── 5.1 A standardized rating schema (score config) + the feedback helper ────
# csat_config() + record_feedback() live in app/feedback.py
from app.feedback import csat_config, record_feedback
CSAT_CONFIG = csat_config()
print("score config ready:", CSAT_CONFIG.name, CSAT_CONFIG.id)


In [ ]:
# ── 5.2 The round-trip, end to end ───────────────────────────────────────────
request_id = "req-2026-07-20-000123"       # your app's own id (chat message id, API request id, …)

result = await run_agent("How do I get an interest certificate for my tax filing?", "CUST-1002",
                   session_id="sess-m5-feedback", tags=["module-5"],
                   trace_seed=request_id)   # ← the trace is CREATED under the derived id
print(result["answer"][:250], "\n")

# … minutes later, in a different process, the user clicks 👍 and rates 4/5:
tid = record_feedback(request_id, thumbs_up=True, comment="Clear steps, thanks!", csat="4-good")
print("scored trace:", tid, "==", result["trace_id"], "→", tid == result["trace_id"])

# user changes their mind → SAME score_id → overwrite, not duplicate
record_feedback(request_id, thumbs_up=False, comment="Actually it never told me the charges.")
print("re-vote sent — same score id, value overwritten to 0")

In [ ]:
# ── pull the scores back  ───────────
scores = lf.api.scores.get_many(name="user-feedback", limit=10)
for s in scores.data[:5]:
    print(f"{s.name}={s.value} ({s.data_type}, source={s.source}) trace={s.trace_id[:12]}… comment={s.comment!r}")
print("\n✅ exactly ONE user-feedback score exists for our trace (the re-vote overwrote), value 0.")

### ✅ CHECKPOINT — feedback in the UI
- Open the module-5 trace: its **Scores tab** shows `user-feedback = 0` (BOOLEAN, source API, with the comment)
and `csat = 4-good` (CATEGORICAL, validated against the config).
- In the Tracing table, filter **Scores ▸ user-feedback < 1** — that's your "unhappy customers" worklist, and in Module 9 the average of this score becomes an **alerting signal**.
